# GMM Exploration on the Michelson data

Now that we have the Michelson data in a good position, we can start working on the GMM to help quantify some distance, with some additions. 

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from sklearn.mixture import GaussianMixture
import helpers.generate_planet_list as helpers
import helpers.generate_galaxy_data as galaxy_helpers
from species import SpeciesInit
from species.data.database import Database
from species.read.read_model import ReadModel
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

SpeciesInit()
db = Database()

def balanced_accuracy_score(y_true, y_pred):
    recall_0 = np.sum((y_true == 0) & (y_pred == 0)) / np.sum(y_true == 0)
    recall_1 = np.sum((y_true == 1) & (y_pred == 1)) / np.sum(y_true == 1)
    balanced_accuracy = (recall_0 + recall_1) / 2
    return balanced_accuracy


db.add_model('sonora-bobcat', wavel_range=(0.61, 17.0), wavel_sampling=1000.0)

bobcat_model = ReadModel('sonora-bobcat')
bobcat_planets = helpers.generate_planet_arrays(bobcat_model, radius_range=(0.6, 1.3), distance=10, num_samples=400)

bobcat_planets = helpers.update_planet_flux_and_magnitude(bobcat_model, bobcat_planets, ['JWST/MIRI.F1065C', 'JWST/MIRI.F1140C', 'JWST/MIRI.F1550C'])
bobcat_planets = helpers.add_color_columns(bobcat_planets, ['F1065C', 'F1140C', 'F1550C'])
names, colors = helpers.color_color_matrix(bobcat_planets)

AGN_00_data = galaxy_helpers.galaxy_color_color_data_k15('./galaxy-data/K15_templates/MIR_library/MIR0.0.txt')
AGN_05_data = galaxy_helpers.galaxy_color_color_data_k15('./galaxy-data/K15_templates/MIR_library/MIR0.5.txt')
AGN_10_data = galaxy_helpers.galaxy_color_color_data_k15('./galaxy-data/K15_templates/MIR_library/MIR1.0.txt')
# get all of the dwarf planet colors for the first color

d_color_1 = colors[:, 0]
d_color_2 = colors[:, 1]
d_color_3 = colors[:, 2]

g_00_color_1 = AGN_00_data[names[0]]
g_05_color_1 = AGN_05_data[names[0]]
g_10_color_1 = AGN_10_data[names[0]]

g_00_color_2 = AGN_00_data[names[1]]
g_05_color_2 = AGN_05_data[names[1]]
g_10_color_2 = AGN_10_data[names[1]]

g_00_color_3 = AGN_00_data[names[2]]
g_05_color_3 = AGN_05_data[names[2]]
g_10_color_3 = AGN_10_data[names[2]]

all_g_color_1 = np.concatenate((g_00_color_1, g_05_color_1, g_10_color_1))
all_g_color_2 = np.concatenate((g_00_color_2, g_05_color_2, g_10_color_2))
all_g_color_3 = np.concatenate((g_00_color_3, g_05_color_3, g_10_color_3))

all_color_1 = np.concatenate((d_color_1, all_g_color_1))
all_color_2 = np.concatenate((d_color_2, all_g_color_2))
all_color_3 = np.concatenate((d_color_3, all_g_color_3))

X = np.column_stack((all_color_1, all_color_2))
Y = np.concatenate((np.zeros(len(d_color_1)), np.ones(len(all_g_color_1))))

gm = GaussianMixture(n_components=2, covariance_type="full", random_state=0).fit(X)
pred = gm.predict(X)

# We calculate the balanced accuracy score for both, because we're resolving a naming convention, not tuning anything. The labels are arbitrary, so we want to make sure we get the best score regardless of which label is assigned to which cluster.
score = max(balanced_accuracy_score(Y, pred), balanced_accuracy_score(Y, 1 - pred))
cm = confusion_matrix(Y, pred)

if balanced_accuracy_score(Y, pred) < balanced_accuracy_score(Y, 1 - pred):
    cm = confusion_matrix(Y, 1 - pred)
print(f"Confusion matrix:\n{cm}")

print(f"Balanced accuracy score X2: {score:.4f}")

X3 = np.column_stack((all_color_1, all_color_2, all_color_3))
gm3 = GaussianMixture(n_components=2, covariance_type="full", random_state=0).fit(X3)
pred3 = gm3.predict(X3)

score3 = max(balanced_accuracy_score(Y, pred3), balanced_accuracy_score(Y, 1 - pred3))
print(f"Balanced accuracy score X3: {score3:.4f}")

print(X3.shape)                                  # expect (n, 3)
print(np.linalg.matrix_rank(X3 - X3.mean(axis=0)))  # expect 2, not 3




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
species

Version: 0.10.4
Working folder: /Users/mitchellfoote/Documents/Source/research/dwarfhunt/michelson-repro

Configuration file: /Users/mitchellfoote/Documents/Source/research/dwarfhunt/michelson-repro/species_config.ini
Database file: species_database.hdf5
Data folder: data

Configuration settings:
   - Database: species_database.hdf5
   - Data folder: data
   - Magnitude of Vega: 0.03

Multiprocessing: mpi4py not installed

-------------------------
Add grid of model spectra
-------------------------

Database tag: sonora-bobcat
Model name: Sonora Bobcat

Unpacking 1131/1131 model spectra from Sonora Bobcat (392 MB)... [DONE]

Please cite Marley et al. (2021) when using Sonora Bobcat in a publication
Reference URL: https://zenodo.org/record/5063476

Wavelength range (um) = 0.61 - 17.0
Sampling (lambda/d_lambda) = 1000.0
Teff range (K) = 200 - 2400

Adding Sonora Bobcat model spectra... data/

/opt/homebrew/Caskroom/miniconda/base/envs/dwarfhunt/lib/python3.11/site-packages/species/util/data_util.py:413: RuntimeWarning: divide by zero encountered in log10
  flux = np.log10(flux)


Confusion matrix:
[[ 56  44]
 [ 40 230]]
Balanced accuracy score X2: 0.7059
Balanced accuracy score X3: 0.7059
(370, 3)
2


/opt/homebrew/Caskroom/miniconda/base/envs/dwarfhunt/lib/python3.11/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/Caskroom/miniconda/base/envs/dwarfhunt/lib/python3.11/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/Caskroom/miniconda/base/envs/dwarfhunt/lib/python3.11/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
